In [0]:
%run ../config/config

In [0]:
!pip install jinja2==3.1.6

In [0]:
# Imports estándar
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from jinja2 import Environment, FileSystemLoader
from datetime import datetime
import sys
import time

# Configurar path del proyecto
sys.path.insert(0, str(PROJECT_PATH))

# Imports de módulos reportes
from utils.reporting import HTMLReportGenerator
from utils.reporting import ChartGenerator
from utils.reporting import TableBuilder
from utils.reporting import HTMLFormatter

# Imports de utilidades
from utils.logger import MLOpsLogger
from utils.metadata_serializer import MetadataSerializer

import mlflow
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

print("Imports completados exitosamente")

In [0]:
# Logger para generación de reportes
logger = MLOpsLogger(
    name='04_quality_inputs',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': 'generate_reports_from_metadata'
    }
)

In [0]:
mlflow.set_registry_uri("databricks")
logger.info(f"MLflow registry URI: {mlflow.get_tracking_uri()}")

In [0]:
# =============================================================================
# CONFIGURACIÓN DE GENERACIÓN DE GRÁFICOS
# =============================================================================

# Flag para controlar la generación de gráficos de distribución
GENERATE_DISTRIBUTION_CHARTS = False

# Flag para controlar la generación de gráficos de drift
GENERATE_DRIFT_CHARTS = False

print(f"Configuración de gráficos:")
print(f"  - Gráficos de distribución: {'ACTIVADOS' if GENERATE_DISTRIBUTION_CHARTS else 'DESACTIVADOS'}")
print(f"  - Gráficos de drift: {'ACTIVADOS' if GENERATE_DRIFT_CHARTS else 'DESACTIVADOS'}")

In [0]:
# =============================================================================
# CARGAR METADATOS DESDE JSONs INDIVIDUALES
# =============================================================================
# Los notebooks 02_engine_quality y 03_engine_drift generan archivos
# separados en metadata/. Este notebook los carga individualmente.
# =============================================================================

try:
    logger.info("=" * 60)
    logger.info("CARGANDO METADATOS DESDE JSONs INDIVIDUALES")
    logger.info("=" * 60)

    start_time = time.time()

    metadata_dir = os.path.join(TEMP_REPORTS_INPUT_PATH, "metadata")

    if not os.path.exists(metadata_dir):
        raise FileNotFoundError(
            f"No se encontró el directorio de metadatos: {metadata_dir}\n"
            f"Asegúrate de ejecutar primero 02_engine_quality y 03_engine_drift"
        )

    # Listar archivos disponibles
    available_files = os.listdir(metadata_dir)
    logger.info(f"Archivos en metadata/: {available_files}")

    # =========================================================================
    # 1) QUALITY METADATA (bundle del notebook 02)
    # =========================================================================
    quality_bundle_path = os.path.join(metadata_dir, "quality_metadata.json")
    # Fallback: intentar con full_metadata.json si existe (compatibilidad)
    if not os.path.exists(quality_bundle_path):
        quality_bundle_path = os.path.join(metadata_dir, "full_metadata.json")

    if not os.path.exists(quality_bundle_path):
        raise FileNotFoundError(
            f"No se encontró quality_metadata.json ni full_metadata.json en {metadata_dir}\n"
            f"Ejecuta primero 02_engine_quality.ipynb"
        )

    logger.info(f"Cargando quality bundle: {quality_bundle_path}")
    metadata_bundle = MetadataSerializer.load_metadata_bundle(quality_bundle_path)

    # Extraer componentes del quality bundle
    metadata_info = metadata_bundle['metadata']
    current_metrics_data = metadata_bundle['current_metrics']
    evolutive_metrics_data = metadata_bundle['evolutive_metrics']
    additional_metadata = metadata_bundle.get('additional', {})
    distribution_data_path = metadata_bundle['paths']['distribution_data']

    logger.info(f" ✓ Quality bundle cargado")
    logger.info(f"   Mes: {metadata_info['mes_vta']} | Registros: {metadata_info['total_records']:,} | Vars: {metadata_info['variables_count']}")

    # Convertir current_metrics a DataFrames
    current_metrics = {
        'summary_metrics': pd.DataFrame(current_metrics_data.get('summary_metrics', [])),
        'variable_metrics': pd.DataFrame(current_metrics_data.get('variable_metrics', [])),
        'variable_types': current_metrics_data.get('variable_types', {}),
        'kpis': current_metrics_data.get('kpis', {})
    }
    logger.info(f"   Variables en current_metrics: {len(current_metrics['variable_metrics'])}")

    # Reconstruir evolutive_metrics con índice
    evolutive_metrics = {}
    for metric_name, data in evolutive_metrics_data.items():
        df = pd.DataFrame(data)

        # Buscar columna Variable (puede ser 'Variable' o '("Variable", "")')
        variable_col = None
        for col in df.columns:
            col_str = str(col)
            if col_str == 'Variable' or col_str == '("Variable", "")':
                variable_col = col
                break

        if variable_col is not None:
            df = df.set_index(variable_col)
            df.index.name = 'Variable'  # Normalizar nombre del índice
        evolutive_metrics[metric_name] = df

    logger.info(f"   Evolutive metrics: {len(evolutive_metrics)} métricas")

    # Info adicional
    meses_historicos_count = additional_metadata.get('meses_historicos', 0)
    chart_bins = additional_metadata.get('chart_bins', 30)
    rows_per_page = additional_metadata.get('rows_per_page', 25)

    # =========================================================================
    # 2) DRIFT METADATA (del notebook 03)
    # =========================================================================
    drift_results = None
    variable_comparison = None
    semaphore_kpis = None
    semaphore_config_drift = None

    # Intentar cargar drift_metrics.json (generado por serialize_drift_metrics)
    drift_metrics_path = os.path.join(metadata_dir, "drift_metrics.json")
    drift_bundle_path = os.path.join(metadata_dir, "drift_metadata.json")

    if os.path.exists(drift_metrics_path):
        logger.info(f"Cargando drift metrics: {drift_metrics_path}")
        with open(drift_metrics_path, 'r', encoding='utf-8') as f:
            drift_data = json.load(f)

        # Extraer componentes del drift
        drift_key = 'drift_results' if 'drift_results' in drift_data else 'drift_metrics'
        if drift_data.get(drift_key):
            drift_results = pd.DataFrame(drift_data[drift_key])
            logger.info(f"  ✓ Drift results: {len(drift_results)} variables")

        # Variable comparison
        var_comp_data = drift_data.get('variable_comparison', [])
        if isinstance(var_comp_data, dict) and var_comp_data:
            try:
                variable_comparison = pd.DataFrame(var_comp_data)
            except ValueError:
                variable_comparison = var_comp_data
        elif isinstance(var_comp_data, list) and var_comp_data:
            variable_comparison = pd.DataFrame(var_comp_data)
        else:
            variable_comparison = None

        semaphore_kpis = drift_data.get('semaphore_kpis', {})
        semaphore_config_drift = drift_data.get('semaphore_config', {})

    elif os.path.exists(drift_bundle_path):
        # Fallback: cargar desde drift_metadata.json (el bundle manual)
        logger.info(f"Cargando drift bundle: {drift_bundle_path}")
        with open(drift_bundle_path, 'r', encoding='utf-8') as f:
            drift_bundle = json.load(f)

        drift_summary = drift_bundle.get('drift_summary', {})
        semaphore_kpis = drift_summary.get('semaphore_kpis', {})
        logger.info(f" ✓ Drift bundle cargado (resumen solamente, sin detalle por variable)")
        logger.warning(f"   ⚠️ drift_metrics.json no encontrado - el reporte de drift tendrá info limitada")
    else:
        logger.info("  🚫 No hay archivos de drift disponibles")

    # =========================================================================
    # 3) EVOLUTIVE SEMAPHORES (opcionales, del notebook 02)
    # =========================================================================
    evolutive_semaphore_data = None
    evolutive_kpis_from_file = None

    evol_sem_path = os.path.join(metadata_dir, "evolutive_semaphores.json")
    if os.path.exists(evol_sem_path):
        logger.info(f"Cargando evolutive semaphores: {evol_sem_path}")
        with open(evol_sem_path, 'r', encoding='utf-8') as f:
            evol_sem_data = json.load(f)

        if evol_sem_data.get('evolutive_semaphores'):
            evolutive_semaphore_data = pd.DataFrame(evol_sem_data['evolutive_semaphores'])
            evolutive_kpis_from_file = evol_sem_data.get('evolutive_kpis', {})
            logger.info(f"  ✓ Evolutive semaphores cargados")

    logger.info(f"\n✓ Todos los metadatos cargados exitosamente")

except Exception as e:
    logger.log_exception(
        operation='load_metadata',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# GENERAR GRÁFICOS DE DISTRIBUCIÓN DESDE METADATOS
# =============================================================================

try:
    if GENERATE_DISTRIBUTION_CHARTS:
        logger.info("=" * 60)
        logger.info("GENERANDO GRÁFICOS DE DISTRIBUCIÓN")
        logger.info("=" * 60)

        charts_start = time.time()

        logger.info(f"Cargando datos de distribución desde: {distribution_data_path}")

        with open(distribution_data_path, 'r', encoding='utf-8') as f:
            distribution_data = json.load(f)

        logger.info(f"✓ Datos de distribución cargados: {len(distribution_data)} variables")

        def fig_to_base64(fig):
            import base64
            from io import BytesIO
            buf = BytesIO()
            fig.savefig(buf, format='png', bbox_inches='tight', dpi=100)
            buf.seek(0)
            return base64.b64encode(buf.getvalue()).decode('utf-8')

        distribution_charts = {}

        for var_name, var_data in distribution_data.items():
            try:
                bin_edges = var_data['bin_edges']
                counts = var_data['counts']
                stats = var_data['stats']

                fig, ax = plt.subplots(figsize=(10, 6))

                bin_centers = [(bin_edges[i] + bin_edges[i+1]) / 2 for i in range(len(bin_edges)-1)]
                bin_widths = [bin_edges[i+1] - bin_edges[i] for i in range(len(bin_edges)-1)]
                                ax.bar(bin_centers, counts, width=bin_widths, edgecolor='k', alpha=0.7)
                
                ax.set_title(f"Distribución de {var_name}")
                ax.set_xlabel(var_name)
                ax.set_ylabel("Frecuencia")

                stats_text = f"Mean: {stats['mean']:.2f}\nStd: {stats['std']:.2f}\nCount: {stats['count']:,}"
                ax.text(0.02, 0.98, stats_text, transform=ax.transAxes,
                        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

                plt.tight_layout()

                b64_string = f"data:image/png;base64,{fig_to_base64(fig)}"
                distribution_charts[f"Histograma_{var_name}"] = b64_string
                plt.close(fig)

            except Exception as col_error:
                logger.warning(f"  Error al generar gráfico para {var_name}: {str(col_error)}")
                continue

        charts_duration = time.time() - charts_start
        logger.info(f"✓ {len(distribution_charts)} gráficos generados en {charts_duration:.2f}s")
    else:
        logger.info("📊 Gráficos de distribución desactivados (GENERATE_DISTRIBUTION_CHARTS=False)")
        distribution_charts = {}
        charts_duration = 0

except Exception as e:
    logger.log_exception(
        operation='generate_charts',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# GENERAR REPORTE HTML DE CALIDAD
# =============================================================================

try:
    logger.info("=" * 60)
    logger.info("GENERANDO REPORTE HTML DE CALIDAD")
    logger.info("=" * 60)

    quality_report_start = time.time()

    os.makedirs(TEMP_REPORTS_INPUT_PATH, exist_ok=True)

    quality_report_path = os.path.join(TEMP_REPORTS_INPUT_PATH, "input_quality_report.html")

    # Template path
    full_template_path = os.path.join(TEMPLATE_REPORT_PATH, 'report_template.html')

    if not os.path.exists(full_template_path):
        raise FileNotFoundError(f"Template no encontrado: {full_template_path}")

    # Cargar configuración de reportes
    from utils.config_loader import ConfigLoader
    config_loader = ConfigLoader(QUALITY_CONFIG)
    all_configs = config_loader.get_all_input_quality_config()
    report_settings = all_configs['reports']
    evolutive_semaforo_thresholds = all_configs['evolutive_thresholds']

    titulo_quality_report = report_settings.get(
        'titulo_quality',
        'Reporte de Calidad de Datos Inputs'
    )

    # Instanciar generador de reportes
    quality_report_generator = HTMLReportGenerator(full_template_path)

    # Generar reporte de calidad
    quality_report_generator.generate_quality_report(
        path_out=quality_report_path,
        titulo=titulo_quality_report,
        current_metrics=current_metrics,
        table_name=metadata_info['table_name'],
        variable_types=current_metrics.get('variable_types', {}),
        evolutive_metrics=evolutive_metrics,
        distribution_charts=distribution_charts,
        evolutive_semaforo_thresholds=evolutive_semaforo_thresholds,
        rows_per_page=rows_per_page
    )

    quality_report_duration = time.time() - quality_report_start

    logger.info(f"✓ Reporte de calidad generado en {quality_report_duration:.2f}s")
    logger.info(f"  Ubicación: {quality_report_path}")
    logger.info(f"  Tamaño: {os.path.getsize(quality_report_path) / 1024:.2f} KB")

except Exception as e:
    logger.log_exception(
        operation='generate_quality_report',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# GENERAR REPORTE HTML DE DRIFT
# =============================================================================

try:
    drift_report_path = None
    drift_report_duration = 0

    if drift_results is not None and not drift_results.empty:
        logger.info("=" * 60)
        logger.info("GENERANDO REPORTE HTML DE DRIFT")
        logger.info("=" * 60)

        drift_report_start = time.time()

        drift_report_path = os.path.join(TEMP_REPORTS_INPUT_PATH, "input_drift_report.html")

        titulo_drift_report = report_settings.get(
            'titulo_drift',
            'Reporte de Data Drift - Inputs'
        )
        drift_metrics_to_consider = all_configs['drift_default_metrics']

        # Gráficos de drift
        if GENERATE_DRIFT_CHARTS:
            logger.info("Generando gráficos de drift...")
            drift_distribution_charts = {}
        else:
            logger.info("📊 Gráficos de drift desactivados (GENERATE_DRIFT_CHARTS=False)")
            drift_distribution_charts = {}

        # Instanciar generador de reportes para drift
        drift_report_generator = HTMLReportGenerator(full_template_path)

        # Generar reporte de drift
        drift_report_generator.generate_drift_report(
            path_out=drift_report_path,
            titulo=titulo_drift_report,
            drift_results=drift_results,
            variable_comparison=variable_comparison if isinstance(variable_comparison, pd.DataFrame) and not variable_comparison.empty else pd.DataFrame(),
            distribution_charts=drift_distribution_charts,
            table_name=metadata_info['table_name'],
            variable_types=current_metrics.get('variable_types', {}),
            semaphore_kpis=semaphore_kpis,
            semaphore_config=semaphore_config_drift,
            metrics_to_consider=drift_metrics_to_consider
        )

        drift_report_duration = time.time() - drift_report_start

        logger.info(f"✓ Reporte de drift generado en {drift_report_duration:.2f}s")
        logger.info(f"  Ubicación: {drift_report_path}")
        logger.info(f"  Tamaño: {os.path.getsize(drift_report_path) / 1024:.2f} KB")
    else:
        logger.info("🚫 No se generará reporte de drift (sin datos válidos)")

except Exception as e:
    logger.log_exception(
        operation='generate_drift_report',
        exception=e,
        should_raise=True
    )

In [0]:
# =============================================================================
# VERIFICACIÓN Y RESUMEN FINAL
# =============================================================================
try:
    total_duration = time.time() - start_time

    logger.info("=" * 60)
    logger.info("GENERACIÓN DE REPORTES COMPLETADA")
    logger.info("=" * 60)
    logger.info(f"Duration total: {total_duration:.2f}s")
    logger.info(f"REPORTES GENERADOS:")
    logger.info(f"  1. Reporte de Calidad: {quality_report_path}")
    if drift_report_path:
        logger.info(f"  2. Reporte de Drift: {drift_report_path}")

    # Verificar archivos
    quality_exists = os.path.exists(quality_report_path)
    logger.info(f"VERIFICACIÓN:")
    logger.info(f"  ✓ Reporte de Calidad: {'OK' if quality_exists else 'ERROR'}")

    if drift_report_path:
        drift_exists = os.path.exists(drift_report_path)
        logger.info(f"  ✓ Reporte de Drift: {'OK' if drift_exists else 'ERROR'}")

    logger.info("=" * 60)
    logger.info("✓ PROCESO COMPLETADO EXITOSAMENTE")
    logger.info("=" * 60)

    logger.log_performance(
        operation='generate_reports_complete',
        duration=total_duration,
        quality_report_time=quality_report_duration,
        drift_report_time=drift_report_duration,
        charts_time=charts_duration
    )

except Exception as e:
    logger.log_exception(
        operation='final_verification',
        exception=e,
        should_raise=False
    )

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()